# PRISM-Endo

## Milestone 2 – Instrument Segmentation

### Objective

The objective of this notebook is to evaluate and integrate the SIS-PT-SAM model for surgical instrument segmentation. This notebook focuses only on identifying surgical instruments in endoscopic images. Real-time tracking, risk prediction, and alert generation will be implemented in later milestones.

## Why SIS-PT-SAM?

PRISM-Endo requires accurate segmentation of surgical instruments before any higher-level analysis can be performed.

SIS-PT-SAM was selected because:

- It is designed specifically for surgical instrument segmentation.
- It is based on the Segment Anything Model (SAM), which provides strong segmentation capabilities.
- It supports fine-tuning on surgical datasets.
- It can serve as the foundation for future modules such as tool tracking, phase recognition, and risk prediction.

Instrument segmentation is the first stage of the PRISM-Endo pipeline.

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
!git clone https://github.com/zijianwu1231/SIS-PT-SAM.git

fatal: destination path 'SIS-PT-SAM' already exists and is not an empty directory.


In [27]:
import os

print(os.listdir('/content'))

['.config', 'drive', 'SIS-PT-SAM', 'sample_data']


In [28]:
%cd /content/SIS-PT-SAM

/content/SIS-PT-SAM


In [29]:
!pwd

/content/SIS-PT-SAM


In [30]:
!ls -la

total 120
drwxr-xr-x 11 root root  4096 Aug 14 13:50 .
drwxr-xr-x  1 root root  4096 Aug 14 13:29 ..
drwxr-xr-x  2 root root  4096 Aug 14 13:29 assets
drwxr-xr-x  2 root root  4096 Aug 14 13:30 ckpts
drwxr-xr-x  7 root root  4096 Aug 14 13:43 cotracker
-rw-r--r--  1 root root 17235 Aug 14 13:29 dataset.py
-rw-r--r--  1 root root  1767 Aug 14 13:29 evaluate.py
drwxr-xr-x  8 root root  4096 Aug 14 13:29 .git
drwxr-xr-x  5 root root  4096 Aug 14 13:30 mobile_sam
-rw-r--r--  1 root root 22291 Aug 14 13:29 online_demo.py
drwxr-xr-x  2 root root  4096 Aug 14 13:43 __pycache__
-rw-r--r--  1 root root  5873 Aug 14 13:29 README.md
-rw-r--r--  1 root root  1200 Aug 14 13:29 requirements.txt
drwxr-xr-x  4 root root  4096 Aug 14 13:29 segment_anything
drwxr-xr-x  8 root root  4096 Aug 14 13:50 SIS-PT-SAM
-rw-r--r--  1 root root  3445 Aug 14 13:29 surgical_tool_sam.py
-rw-r--r--  1 root root  8711 Aug 14 13:29 train.py
drwxr-xr-x  2 root root  4096 Aug 14 13:29 utils


In [31]:
!sed -n '1,200p' README.md

# Augmenting Efficient Surgical Instrument Segmentation in Video with Point Tracking and Segment Anything (SIS-PT-SAM)

## News
This work won the <strong>Outstanding Paper Award</strong> at MICCAI 2024 AE-CAI Workshop!

The work was accepted by MICCAI 2024 AE-CAI Workshop!

PyTorch implementation of the SIS-PT-SAM. Inference speed achieves 25+/80+ FPS on single RTX 4060/4090 GPU. Use point prompts for full fine-tuning MobileSAM.

[[arXiv](https://arxiv.org/abs/2403.08003)]

<!-- ## Description

An in-depth paragraph about your project and overview of use. -->

<img src="assets/cholecseg8k_demo_8.gif" height="180"> <img src="assets/stir_demo.gif" height="180"> <img src="assets/ucl_demo_13.gif" height="180">
## Getting Started
### Dependencies

* Python 3.11
* torch 2.1.2
* torchvision 0.16.2

### Environment Setup
Create a Conda environment:
```
conda create --name sis-pt-sam python=3.11
```

Activate the Conda environment and run the following command to install environment
```
pip ins

In [32]:
!cat requirements.txt

certifi==2024.6.2
charset-normalizer==3.3.2
contourpy==1.2.1
cycler==0.12.1
filelock==3.15.4
fonttools==4.53.0
fsspec==2024.6.0
huggingface-hub==0.23.4
idna==3.7
imageio==2.34.1
imageio-ffmpeg==0.5.1
Jinja2==3.1.4
joblib==1.4.2
kiwisolver==1.4.5
lazy_loader==0.4
MarkupSafe==2.1.5
matplotlib==3.9.0
monai==1.3.1
mpmath==1.3.0
networkx==3.3
numpy==1.26.1
nvidia-cublas-cu12==12.1.3.1
nvidia-cuda-cupti-cu12==12.1.105
nvidia-cuda-nvrtc-cu12==12.1.105
nvidia-cuda-runtime-cu12==12.1.105
nvidia-cudnn-cu12==8.9.2.26
nvidia-cufft-cu12==11.0.2.54
nvidia-curand-cu12==10.3.2.106
nvidia-cusolver-cu12==11.4.5.107
nvidia-cusparse-cu12==12.1.0.106
nvidia-nccl-cu12==2.18.1
nvidia-nvjitlink-cu12==12.5.40
nvidia-nvtx-cu12==12.1.105
opencv-python==4.10.0.84
packaging==24.1
pillow==10.3.0
psutil==6.0.0
pyparsing==3.1.2
python-dateutil==2.9.0.post0
PyYAML==6.0.1
requests==2.32.3
safetensors==0.4.3
scikit-image==0.24.0
scikit-learn==1.5.0
scikit-learn-extra==0.3.0
scipy==1.13.1
segment-anything-hq==0.3
six==1.

In [1]:

import torch
import torchvision
import cv2
import numpy as np
import matplotlib

print("Torch       :", torch.__version__)
print("TorchVision :", torchvision.__version__)
print("OpenCV      :", cv2.__version__)
print("NumPy       :", np.__version__)
print("Matplotlib  :", matplotlib.__version__)

Torch       : 2.3.1+cu121
TorchVision : 0.18.1+cu121
OpenCV      : 5.0.0
NumPy       : 1.26.1
Matplotlib  : 3.10.0


In [2]:
import torch

print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("PyTorch is NOT using the GPU.")

CUDA Available : True
GPU : Tesla T4


In [3]:
!nvidia-smi

Fri Aug 14 14:01:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

True
Tesla T4


In [5]:
!grep -n "class " surgical_tool_sam.py
!grep -n "def forward" surgical_tool_sam.py

grep: surgical_tool_sam.py: No such file or directory
grep: surgical_tool_sam.py: No such file or directory


In [ ]:
"""Now we need to know:

"How do I start the engine?"

That's exactly what the repository's train.py or online_demo.py will show us."""
!grep -R "SurgicalToolSAM(" -n .

In [ ]:
"""We only need to see how the model is created."""
!sed -n '55,90p' online_demo.py

In [ ]:
!grep -n "mobile_sam_checkpoint" online_demo.py
!grep -n "sam_checkpoint" online_demo.py
"""This will tell us:

where they're loaded from,
what filenames they expect,
where we need to place them."""

In [ ]:
"""Create the expected folder:"""
!mkdir -p ckpts

In [ ]:
!ls ckpts

##Installing
1. MobileSAM checkpoint
2. SIS-PT-SAM pretrained checkpoint

Directly into Colab (install once)

In [ ]:
#!pip install -q gdown

In [ ]:
!gdown --id 1h9qpgzPM6rcY82x8HjLojZk9UQIbpV2C -O ckpts/surgicaltoolsam_best_ff.pth
!ls -lh ckpts

In [ ]:
!wget -O ckpts/mobile_sam.pt \
https://github.com/ChaoningZhang/MobileSAM/raw/master/weights/mobile_sam.pt

!ls -lh ckpts

In [ ]:
!sed -n '1,80p' surgical_tool_sam.py

##Actual model

In [ ]:
import torch
from surgical_tool_sam import SurgicalToolSAM

In [ ]:
model = SurgicalToolSAM(
    ckpt="./ckpts/mobile_sam.pt",
    freeze_image_encoder=True,
    freeze_prompt_encoder=True,
    freeze_mask_decoder=True
)

In [ ]:
import torch

checkpoint = torch.load(
    "./ckpts/surgicaltoolsam_best_ff.pth",
    map_location="cuda"
)

model.load_state_dict(checkpoint["model"])

print("✅ Pretrained weights loaded successfully.")

Google Colab

Clone Repository ✅

Download Checkpoints ✅

Create MobileSAM Architecture ✅

Load SIS-PT-SAM Weights ✅

AI Model Ready 🚀

##Phase 2.1 – Load Your First Surgical Video

In [ ]:
import cv2
import matplotlib.pyplot as plt

frame_path = "/content/drive/MyDrive/Colab Notebooks/PRISM-Endo/video01/video01_00080/frame_80_endo.png"

image = cv2.imread(frame_path)

print("Image loaded:", image is not None)
print("Image shape:", image.shape)

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 7))
plt.imshow(image_rgb)
plt.title("CholecSeg8k - Frame 80")
plt.axis("off")
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt

base_path = "/content/drive/MyDrive/Colab Notebooks/PRISM-Endo/video01/video01_00080"

files = [
    "frame_80_endo.png",
    "frame_80_endo_mask.png",
    "frame_80_endo_color_mask.png",
    "frame_80_endo_watershed_mask.png"
]

for file in files:
    path = f"{base_path}/{file}"
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)

    print(file, "→", img.shape, img.dtype)

In [ ]:
import cv2
import matplotlib.pyplot as plt

base_path = "/content/drive/MyDrive/Colab Notebooks/PRISM-Endo/video01/video01_00080"

files = [
    "frame_80_endo.png",
    "frame_80_endo_mask.png",
    "frame_80_endo_color_mask.png",
    "frame_80_endo_watershed_mask.png"
]

titles = [
    "Original Frame",
    "Mask",
    "Color Mask",
    "Watershed Mask"
]

plt.figure(figsize=(16, 10))

for i, (file, title) in enumerate(zip(files, titles)):
    img = cv2.imread(f"{base_path}/{file}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(2, 2, i + 1)
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import torch

# Paths
base_path = "/content/drive/MyDrive/Colab Notebooks/PRISM-Endo/video01/video01_00080"

image_path = f"{base_path}/frame_80_endo.png"
mask_path = f"{base_path}/frame_80_endo_mask.png"

# Load image
image = cv2.imread(image_path)

# Load ground-truth mask
gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

print("Image:", image.shape)
print("Mask :", gt_mask.shape)

In [ ]:
frame_path = "/content/drive/MyDrive/Colab Notebooks/PRISM-Endo/video01/video01_00080/frame_149_endo.png"

image = cv2.imread(frame_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

print("Image shape:", image.shape)

plt.figure(figsize=(12, 7))
plt.imshow(image_rgb)
plt.title("Frame 149")
plt.axis("off")
plt.show()

In [ ]:
# Load the ground-truth mask for frame 149
mask_path = "/content/drive/MyDrive/Colab Notebooks/PRISM-Endo/video01/video01_00080/frame_149_endo_mask.png"

gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

print("Mask shape:", gt_mask.shape)
print("Unique values:", np.unique(gt_mask))

In [ ]:
# Find coordinates belonging to any non-background region
ys, xs = np.where(gt_mask > 0)

print("Annotated pixels:", len(xs))

# Pick the center point of the annotated region
point_x = int(np.mean(xs))
point_y = int(np.mean(ys))

print("Point prompt:", (point_x, point_y))

In [ ]:
# Create a positive point prompt
point_prompt = torch.tensor(
    [[[409, 265]]],
    dtype=torch.float32,
    device="cuda"
)

# Point label: 1 = positive point
point_label = torch.tensor(
    [[1]],
    dtype=torch.int64,
    device="cuda"
)

# SIS-PT-SAM expects points as (coordinates, labels)
point_prompt = (point_prompt, point_label)

print("Point coordinates:", point_prompt[0])
print("Point label:", point_prompt[1])

In [ ]:
import os
import cv2
import torch
import numpy as np

dataset_root = "/content/drive/MyDrive/Colab Notebooks/PRISM-Endo/video01"

# Find all folders inside video01
frame_folders = sorted([
    os.path.join(dataset_root, f)
    for f in os.listdir(dataset_root)
    if os.path.isdir(os.path.join(dataset_root, f))
])

print("Number of frame folders:", len(frame_folders))
print("First folder:", frame_folders[0])

# Collect original surgical frames
image_paths = []

for folder in frame_folders:
    for filename in sorted(os.listdir(folder)):
        if filename.endswith("_endo.png") and not filename.endswith("_mask.png"):
            image_paths.append(os.path.join(folder, filename))

print("Total surgical frames found:", len(image_paths))
print("First frame:", image_paths[0])

In [ ]:
# Select frame 149 automatically
test_path = "/content/drive/MyDrive/Colab Notebooks/PRISM-Endo/video01/video01_00080/frame_149_endo.png"
test_mask_path = "/content/drive/MyDrive/Colab Notebooks/PRISM-Endo/video01/video01_00080/frame_149_endo_mask.png"

image = cv2.imread(test_path)
gt_mask = cv2.imread(test_mask_path, cv2.IMREAD_GRAYSCALE)

print("Input frame :", image.shape)
print("Ground truth:", gt_mask.shape)

In [ ]:
!grep -n -E "Resize|resize|transform|preprocess|set_image|point_prompt" online_demo.py | head -40

In [ ]:
!sed -n '100,140p' online_demo.py